# IL3.2: Análisis de Trazabilidad y Logs
## Notebook 2: Trazabilidad con trace_id y decisiones paso a paso en Agentes Reales

### Objetivo:
Implementar la correlación de eventos usando un identificador único global conocido como `trace_id`. Esto le permite al desarrollador seguirle la pista a una transacción de principio a fin, especialmente cuando una única solicitud del usuario desencadena múltiples eventos y decisiones internas dentro del agente.

### ¿Qué es un `trace_id`?
Es un identificador único (comúnmente un UUID) que se genera cuando el usuario hace una pregunta. A lo largo del ciclo de vida de esa pregunta (razonamiento, selección de herramientas, ejecución de herramientas, formateo de la respuesta), cada línea de log generada se asocia con este identificador. Así, aunque haya múltiples usuarios interactuando simultáneamente y escribiendo en el mismo archivo de log, es posible filtrar y reconstruir el "hilo de decisión" (decision trail) de una sola consulta.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


c:\Users\realm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM de LangChain configurado.
✅ Agente y herramientas listos.


### Trazabilidad con trace_id en un Agente LLM Real (LangChain)
LangChain cuenta con un sistema robusto de eventos mediante **Callbacks**. Podemos crear un handler personalizado heredando de `BaseCallbackHandler` para registrar de forma automática las decisiones internas del agente real (cuándo inicia, qué herramienta llama, qué devuelve la herramienta, etc.) y asociarlas al `trace_id` de la solicitud actual.


In [2]:
import logging
import uuid
import time
from langchain_core.callbacks import BaseCallbackHandler

# Configuración del logger para escribir en 'agent_trace.log'
logging.basicConfig(
    filename="agent_trace.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)

class TracedAgentCallbackHandler(BaseCallbackHandler):
    def __init__(self, trace_id):
        self.trace_id = trace_id

    def on_chain_start(self, serialized, inputs, **kwargs):
        logging.info(f"[{self.trace_id}] [Chain Start] Iniciando ejecución del agente. Input: {inputs}")

    def on_tool_start(self, serialized, input_str, **kwargs):
        logging.info(f"[{self.trace_id}] [Tool Start] Decisión de usar herramienta: {serialized.get('name', 'unknown')} | Input: {input_str}")

    def on_tool_end(self, output, **kwargs):
        logging.info(f"[{self.trace_id}] [Tool End] Herramienta devolvió resultado: {output}")

    def on_chain_end(self, outputs, **kwargs):
        logging.info(f"[{self.trace_id}] [Chain End] Respuesta final generada por el agente. Output: {outputs}")

    def on_chain_error(self, error, **kwargs):
        logging.error(f"[{self.trace_id}] [Chain Error] Fallo interno: {error}")


### Ejecución del Agente Real con Callback y trace_id
A continuación, envolveremos la ejecución en una función `run_traced_wikipedia_agent` que genera un UUID de traza para cada consulta y le inyecta el callback personalizado al ejecutar `agent_executor.invoke`.


In [3]:
def run_traced_wikipedia_agent(query):
    # Generar el trace_id único para esta consulta
    trace_id = str(uuid.uuid4())
    handler = TracedAgentCallbackHandler(trace_id)
    
    logging.info(f"[{trace_id}] === INICIO DE SOLICITUD DE USUARIO ===")
    logging.info(f"[{trace_id}] Prompt del usuario: {query}")
    
    start_time = time.time()
    try:
        if llm is None:
            # Flujo alternativo simulado en caso de no haber API configurada
            logging.info(f"[{trace_id}] [Simulado] Iniciando flujo simulado de Wikipedia.")
            time.sleep(0.1)
            # Simular ejecución de herramienta
            handler.on_tool_start({"name": "get_wikipedia_summary"}, query)
            time.sleep(0.3)
            result = f"Resumen simulado sobre {query} de wikipedia."
            handler.on_tool_end(result)
            response = {"output": f"El resultado simulado para {query} es: {result}"}
        else:
            # Ejecución real pasando el callback en la configuración
            response = agent_executor.invoke(
                {"input": query},
                config={"callbacks": [handler]}
            )
    except Exception as e:
        logging.error(f"[{trace_id}] Error en la consulta: {e}")
        response = {"output": f"Ocurrió un error: {e}"}
        
    latency = round(time.time() - start_time, 4)
    logging.info(f"[{trace_id}] Latencia total de respuesta: {latency} segundos")
    logging.info(f"[{trace_id}] === FIN DE SOLICITUD DE USUARIO ===")
    
    return trace_id, response.get("output", "")

# Ejecutar un par de consultas que generen caminos de decisión distintos
trace_id_1, res_1 = run_traced_wikipedia_agent("¿Quién fue Nikola Tesla?")
print(f"Consulta 1 (Trace ID: {trace_id_1})\nRespuesta: {res_1}\n")

trace_id_2, res_2 = run_traced_wikipedia_agent("Hola, ¿cómo estás hoy?")
print(f"Consulta 2 (Trace ID: {trace_id_2})\nRespuesta: {res_2}")




> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Nikola Tesla'}`


Nikola Tesla (Smiljan, Imperio austríaco, actual Croacia; 10 de julio de 1856-Nueva York, 7 de enero de 1943) fue un inventor e ingeniero serbio-croata  nacionalizado estadounidense,​​​ cuya visión y descubrimientos sentaron las bases de la civilización tecnológica moderna. Célebre como el inventor del motor electromagnético y el arquitecto del sistema polifásico de corriente alterna (CA), innovaciones que permitieron la electrificación masiva a partir del hito histórico del aprovechamiento hidroeléctrico de las Cataratas del Niágara.​ ​ Obtuvo más de 280 patentes en 26 países, de las cuales 112 fueron en Estados Unidos incluyendo el invento de capacitor eléctrico (condensadores eléctricos) y transformadores de alta tensión.​ 
Su genio fue precursor de la robótica y la automatización; en 1898, con la invención y exhibición de su autómata teledirigido, patentó el primer sistema 

### Filtrado del Log de Trazas
Al leer el log, podemos filtrar las líneas que contienen un `trace_id` en particular para aislar y reconstruir de forma secuencial los pasos que tomó el agente real.


In [5]:
# Reconstruir la traza de la consulta sobre Nikola Tesla (Consulta 1)
target_trace = trace_id_1
print(f"--- Reconstruyendo la decisión para el trace_id: {target_trace} ---")

with open("agent_trace.log", "r", encoding="latin-1") as f:
    for line in f:
        if target_trace in line:
            print(line.strip())


--- Reconstruyendo la decisión para el trace_id: b10015ad-0c62-4089-b088-1cc6237777eb ---
2026-06-05 19:07:59,018 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] === INICIO DE SOLICITUD DE USUARIO ===
2026-06-05 19:07:59,019 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] Prompt del usuario: ¿Quién fue Nikola Tesla?
2026-06-05 19:07:59,024 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] [Chain Start] Iniciando ejecución del agente. Input: {'input': '¿Quién fue Nikola Tesla?'}
2026-06-05 19:07:59,024 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] [Chain Start] Iniciando ejecución del agente. Input: {'input': ''}
2026-06-05 19:07:59,025 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] [Chain Start] Iniciando ejecución del agente. Input: {'input': ''}
2026-06-05 19:07:59,026 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] [Chain Start] Iniciando ejecución del agente. Input: {'input': ''}
2026-06-05 19:07:59,027 | INFO | [b10015ad-0c62-4089-b088-1cc6237777eb] [Chain Start] Iniciando ejecuc

### 🛠️ Reto Práctico (Mini-entrega)

**Instrucciones:**
1. Modifica la clase `TracedAgentCallbackHandler` para que registre una marca de tiempo adicional en milisegundos cuando se inicia una herramienta (`on_tool_start`) y calcule el tiempo exacto que duró la ejecución de dicha herramienta cuando termine (`on_tool_end`), imprimiéndolo en el log persistente.
2. Ejecuta una nueva consulta al agente de Wikipedia usando tu callback modificado.
3. Filtra el log con el `trace_id` de esta nueva consulta y comprueba que se haya guardado y calculado el tiempo de ejecución de la herramienta.


In [ ]:
# Desarrolla tu solución aquí

# 1. Callback Handler modificado

# 2. Ejecución de la nueva consulta

# 3. Lectura y filtrado del log de trazas


### 📝 Preguntas de Análisis
1. **¿Por qué es problemático usar solo marcas de tiempo (timestamps) en lugar de un `trace_id` para reconstruir la historia de una consulta en un servidor web con muchos usuarios activos simultáneamente?**
2. **¿Cómo puede un desarrollador usar un `trace_id` para conectar los logs del backend con los errores reportados en la interfaz gráfica (frontend) del usuario?**
3. **Al usar agentes complejos (por ejemplo, multi-agentes en CrewAI o LangGraph), ¿cómo se podría estructurar un `trace_id` para diferenciar los pasos de los sub-agentes individuales dentro de una misma consulta de nivel superior?**
